In [1]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
from datetime import datetime
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={os.getenv('DB_SERVER')};"
    "DATABASE=Ventas_Comerssia;"
    f"UID={os.getenv('DB_USER')};"
    f"PWD={os.getenv('DB_PASSWORD')};"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [3]:
fecha_corte = pd.to_datetime("2026-10-31")

# Rango de análisis 
fecha_inicio_12m = fecha_corte - pd.DateOffset(months=12)
fecha_inicio_24m = fecha_corte - pd.DateOffset(months=24)

In [4]:
# ===============================
# Datos de clientes y ventas
# ===============================
df_clientes = pd.read_sql(f"""
SELECT Cliente, 
CosechaFecha 
FROM Ventas_Comerssia.dbo.Cliente_Perfil
WHERE CosechaFecha <= '{fecha_corte}'""", engine)
df_clientes["CosechaFecha"] = pd.to_datetime(df_clientes["CosechaFecha"], errors="coerce")

query_ventas = f"""
SELECT Cliente, Fecha, Venta_Neta AS Venta
FROM Ventas_Comerssia.dbo.Ventas_Unificadas
WHERE Fecha BETWEEN '{fecha_inicio_24m:%Y-%m-%d}' AND '{fecha_corte:%Y-%m-%d}'
"""
df_ventas = pd.read_sql(query_ventas, engine)
df_ventas["Fecha"] = pd.to_datetime(df_ventas["Fecha"], errors="coerce")

In [5]:
# ===============================
# 3) Métricas por cliente
# ===============================
# Última compra 
ultima_compra = (
    df_ventas.groupby("Cliente")["Fecha"]
    .max()
    .reset_index()
    .rename(columns={"Fecha": "UltimaCompra"})
)

# Ventas últimos 12 meses (para SegmentoActual)
ventas_12m = (
    df_ventas[df_ventas["Fecha"] >= fecha_inicio_12m]
    .groupby("Cliente")["Venta"]
    .sum()
    .reset_index()
    .rename(columns={"Venta": "Venta12M"})
)

# Ventas 24 meses (para Segmento24M)
ventas_24m = (
    df_ventas.groupby("Cliente")["Venta"]
    .sum()
    .reset_index()
    .rename(columns={"Venta": "Venta24M"})
)

# ===============================
# 4) Merge sobre la base completa de clientes
# ===============================
clientes = df_clientes.merge(ultima_compra, on="Cliente", how="left")
clientes = clientes.merge(ventas_12m, on="Cliente", how="left")
clientes = clientes.merge(ventas_24m, on="Cliente", how="left")


In [6]:
# ===============================
# 5) Función exacta para asignar segmento por monto 
# ===============================
def segmento_por_valor(valor):
    if pd.isna(valor) or valor == 0:
        return "Sin Segmento"
    elif valor > 1_400_000:
        return "Diamante"
    elif valor >= 700_000:
        return "Oro"
    elif valor >= 300_000:
        return "Plata"
    else:
        return "Bronce"

clientes["Segmento12M"] = clientes["Venta12M"].apply(segmento_por_valor)
clientes["Segmento24M"] = clientes["Venta24M"].apply(segmento_por_valor)

# ===============================
# 6) Recencia en días 
# ===============================
# calcular dias desde ultima compra 
clientes["RecenciaDias"] = (fecha_corte - clientes["UltimaCompra"]).dt.days

def calcular_recencia(row):
    # Si no tiene ultima compra -> devolvemos 
    if pd.isna(row["UltimaCompra"]):
        return None

    # Nuevo: basado en cosecha (<= 90 dias)
    if pd.notna(row["CosechaFecha"]):
        diff_cosecha = (fecha_corte - row["CosechaFecha"]).days
        if diff_cosecha <= 90:
            return "Nuevo"

    d = row["RecenciaDias"]

    if pd.isna(d):
        return None
    if d <= 120:
        return "Muy Activo"
    elif d <= 300:
        return "Activo"
    elif d <= 365:
        return "Por Inactivar"
    elif d <= 395:
        return "Churn"
    else:
        return "Inactivo"

clientes["Recencia"] = clientes.apply(calcular_recencia, axis=1)

# ===============================
# 7) Definir Segmento final 
# ===============================
def asignar_segmento_final(row):
    rec = row["Recencia"]
    if rec in ["Nuevo", "Muy Activo", "Activo", "Por Inactivar"]:
        return row["Segmento12M"]
    if rec in ["Churn", "Inactivo"]:
        return row["Segmento24M"]
    # Si no tiene recencia (no compró) -> si tiene ventas24M ==0 -> Sin Segmento
    return "Sin Segmento"

clientes["Segmento"] = clientes.apply(asignar_segmento_final, axis=1)

# ===============================
# 8) Regla final: si Segmento 
# ===============================
clientes.loc[clientes["Segmento"] == "Sin Segmento", "Recencia"] = None

In [7]:
# ===============================
# 9) Resultado final (columns)
# ===============================
resultado = clientes[[
    "Cliente", "CosechaFecha", "UltimaCompra",
    "Venta12M", "Venta24M",
    "Recencia", "Segmento"
]]

# Mostrar ejemplo
print(resultado.head(50))

           Cliente CosechaFecha UltimaCompra   Venta12M    Venta24M  \
0               C0   2024-09-27   2026-09-17  149573.40   454392.23   
1        C00067637   2025-01-28   2025-01-28        NaN   278991.60   
2     C00109775015   2026-02-19   2026-02-19  233277.31   233277.31   
3      C0011842416   2023-01-27          NaT        NaN         NaN   
4      C0011863809   2022-04-13          NaT        NaN         NaN   
5      C0043281329   2022-12-09          NaT        NaN         NaN   
6      C0049162580   2019-12-26          NaT        NaN         NaN   
7        C00515329   2024-05-15          NaT        NaN         NaN   
8       C005524746   2022-10-15          NaT        NaN         NaN   
9       C009734492   2024-06-23          NaT        NaN         NaN   
10       C00997154   2023-02-14          NaT        NaN         NaN   
11     C0101347938   2021-09-17          NaT        NaN         NaN   
12     C0201610333   2020-12-22          NaT        NaN         NaN   
13    

In [8]:
# resultado.to_excel("segmentacion_clientes.xlsx", index=False)

In [10]:
resultado.to_sql("Segmentacion_Clientes", engine, if_exists="replace", index=False)

81